# Treino do modelo do sistema assistivo (TCC)

Ajuste fino do **YOLO26 a partir dos pesos do COCO**, num dataset juntado de 29 classes:

| Fonte | Para quê |
|---|---|
| COCO (subconjunto) | manter e refinar pessoa, cadeira, carro, mesa... |
| Accessibility Barriers | **degrau, meio-fio, escada** |
| HomeObjects-3K + DoorDetect | **porta** |
| ROD | **poste, lixeira, cone, placa, faixa de pedestres** |
| **Imagens da ESP32-CAM** (revisadas pela equipe) | a câmera de verdade |

**Duas rodadas — o notebook descobre sozinho em qual está:**

1. **Sem `esp_revisado.zip` no Drive** → rodada 1, só dados públicos. Gera `assistivo-publico.pt`,
   que já detecta escada, degrau, meio-fio, porta, faixa, poste, lixeira, cone e placa.
2. **Com `esp_revisado.zip`** → rodada 2: ajuste fino **a partir da rodada 1**, com as imagens
   da ESP. Gera `assistivo-final.pt` e a comparação no teste da ESP (a tabela do TCC).

Rode as células em ordem (Shift+Enter). Antes: `Ambiente de execução → Alterar o tipo de ambiente
→ GPU T4`. Os tempos são estimativas — dependem da GPU que o Colab entregar. Deixe a aba aberta:
o Colab gratuito desconecta sessões ociosas. Professor, dataset e pesos ficam salvos no Drive; se
cair, rode de novo 1 a 3 e a 6b.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Drive e configuração
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/deep-vision'   # tudo que precisa sobreviver fica aqui
os.makedirs(DRIVE, exist_ok=True)

ESP_ZIP = f'{DRIVE}/esp_revisado.zip'           # exportação do CVAT (YOLO 1.1), COM as imagens
PUBLICO = f'{DRIVE}/assistivo-publico.pt'       # resultado da rodada 1
PROFESSOR = f'{DRIVE}/runs/professor/weights/best.pt'
TEM_ESP = os.path.exists(ESP_ZIP)

if TEM_ESP and os.path.exists(PUBLICO):
    BASE, EPOCAS_FINAL, NOME = PUBLICO, 20, 'assistivo-final'     # rodada 2: parte da rodada 1
elif TEM_ESP:
    BASE, EPOCAS_FINAL, NOME = 'yolo26s.pt', 40, 'assistivo-final'
else:
    BASE, EPOCAS_FINAL, NOME = 'yolo26s.pt', 40, 'assistivo-publico'  # rodada 1: só dados públicos
EPOCAS_PROFESSOR = 20
IMAGENS_COCO, IMAGENS_ROD = 4000, 3000
DATASET_ZIP = f'{DRIVE}/dataset_{NOME}.zip'     # o dataset montado, para retomar sem refazer

print(f'Rodada: {NOME} | imagens da ESP: {TEM_ESP} | parte de: {BASE} | épocas: {EPOCAS_FINAL}')

In [ ]:
# 3. Código do projeto e dependências (a mesma versão do ultralytics das medições)
!git clone -q https://github.com/JeanLimaa/deep-vision.git /content/deep-vision || git -C /content/deep-vision pull -q
!pip install -q "ultralytics==8.4.160" "pydantic-settings>=2.6"
%cd /content/deep-vision/server
cmd = (f'rm -rf /content/esp && mkdir -p /content/esp && unzip -q "{ESP_ZIP}" -d /content/esp'
       if TEM_ESP else 'echo "Sem imagens da ESP: rodada 1, so dados publicos."')
!{cmd}

## 4. Professor das classes novas (~1 a 1,5 h, só na primeira vez)

Cada dataset rotulou só as próprias classes: a porta que aparece numa foto do dataset de degraus
está **sem rótulo**, e isso ensinaria ao modelo que porta é fundo. Este "professor" aprende só
degrau, meio-fio, escada, porta, faixa, poste, lixeira, cone e placa; na célula 5 ele preenche
essas classes nas imagens das outras fontes. Fica salvo no Drive e é reaproveitado na rodada 2.

In [ ]:
from ultralytics import YOLO

ja_treinado = os.path.exists(PROFESSOR)
cmd = ('echo "Professor ja treinado: reaproveitando."' if ja_treinado else
       f'python training/build_dataset.py build --stage professor --out /content/professor '
       f'--cache /content/cache --rod {IMAGENS_ROD} --overwrite')
!{cmd}
if not ja_treinado:
    YOLO('yolo26s.pt').train(data='/content/professor/data.yaml', epochs=EPOCAS_PROFESSOR,
                             imgsz=640, batch=16, patience=8, project=f'{DRIVE}/runs',
                             name='professor', exist_ok=True)

## 5. Dataset (~40 a 60 min na primeira vez)

Junta tudo. Rótulos automáticos só **acrescentam** objetos das classes que cada fonte não rotulou;
rótulo humano nunca é substituído. 30% das imagens públicas de treino são degradadas como a OV2640
(escuro, borrão, JPEG forte, baixa resolução, cor). Na rodada 2, o **teste** é só de imagens da ESP,
de sessões que não entram no treino. O dataset montado vai para o Drive (~3 GB): se o Colab cair,
esta célula só descompacta. Detalhes em `manifest.csv` e `SOURCES.md` (licenças).

In [ ]:
REFAZER_DATASET = False   # True para montar de novo mesmo com o zip no Drive

if os.path.exists(DATASET_ZIP) and not REFAZER_DATASET:
    cmd = f'rm -rf /content/assistivo && unzip -q "{DATASET_ZIP}" -d /content'
else:
    esp = '--esp /content/esp' if TEM_ESP else ''
    cmd = (f'python training/build_dataset.py build {esp} --out /content/assistivo '
           f'--cache /content/cache --coco {IMAGENS_COCO} --rod {IMAGENS_ROD} '
           f'--teachers yolo26l.pt yolo26l-objv1-150.pt "{PROFESSOR}" --overwrite '
           f'&& cd /content && zip -qr -0 "{DATASET_ZIP}" assistivo '
           f'&& cp assistivo/manifest.csv assistivo/SOURCES.md "{DRIVE}/"')
!{cmd}

## 6. Treino (algumas horas)

Rodada 1: parte dos pesos do COCO — o ultralytics casa as 20 classes do COCO pelo nome e
reaproveita a camada de classificação delas ("Transferred 708/708"); só as classes novas começam
do zero. Rodada 2: parte do modelo da rodada 1, que já conhece as 29 classes, e só se adapta à
câmera da ESP — por isso menos épocas.

In [ ]:
YOLO(BASE).train(data='/content/assistivo/data.yaml', epochs=EPOCAS_FINAL, imgsz=640, batch=16,
                 patience=15, cos_lr=True, close_mosaic=10, hsv_v=0.5, degrees=5,
                 project=f'{DRIVE}/runs', name=NOME, exist_ok=True, save_period=5)

In [ ]:
# 6b. Só se o Colab caiu no meio do treino: rode 1, 2, 3 e 5 (que agora só descompacta) e esta.
YOLO(f'{DRIVE}/runs/{NOME}/weights/last.pt').train(resume=True)

## 7. Avaliação

As três configurações no **mesmo** conjunto: na rodada 2, o teste da ESP (sessões que o modelo nunca
viu) — a tabela de antes e depois do TCC; na rodada 1, a validação pública.

1. `yolo26l` só COCO — linha de base;
2. `yolo26l` + Objects365 — linha de base com classes extras, sem treino;
3. o modelo treinado.

`pipeline_ablation.py` mede como o servidor (limiar 0,35, perfil de mobilidade, erros separados em
fantasma, rótulo trocado e mal localizado); o `val` do ultralytics dá o mAP oficial por classe.

In [ ]:
import subprocess

TREINADO = f'{DRIVE}/runs/{NOME}/weights/best.pt'
SPLIT = 'test' if TEM_ESP else 'val'
QUADRO_DA_PLACA = ['--as-is'] if TEM_ESP else []  # imagens da ESP ja sao o JPEG da placa

def avaliar(nome, pesos, extra=()):
    cmd = ['python', 'training/pipeline_ablation.py', '--data', '/content/assistivo',
           '--split', SPLIT, *QUADRO_DA_PLACA, '--only', 'original',
           '--names', '/content/assistivo/data.yaml', '--weights', pesos, '--device', 'cuda',
           '--json', f'{DRIVE}/resultado_{NOME}_{nome}.json']
    if extra:
        cmd += ['--extra', *extra]
    print(f'== {nome}')
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)

avaliar('coco', 'yolo26l.pt')
avaliar('coco_objects365', 'yolo26l.pt', ['yolo26l-objv1-150.pt'])
avaliar('treinado', TREINADO)

metricas = YOLO(TREINADO).val(data='/content/assistivo/data.yaml', split=SPLIT,
                              project=f'{DRIVE}/runs', name=f'{NOME}_{SPLIT}')
print(f'mAP50 = {metricas.box.map50:.3f}   mAP50-95 = {metricas.box.map:.3f}')

## 8. Exportar para o servidor

Baixe o arquivo do Drive para a pasta `models/` do projeto e, no `server/.env`:

```
AVS_VISION__MODEL_PATH=assistivo-publico.pt      # rodada 1
AVS_VISION__MODEL_PATH=assistivo-final.pt        # rodada 2
```

**Sem** `AVS_VISION__EXTRA_MODEL_PATHS`: o modelo treinado já tem todas as classes.

In [ ]:
import shutil

shutil.copy(f'{DRIVE}/runs/{NOME}/weights/best.pt', f'{DRIVE}/{NOME}.pt')
print('Pronto:', f'{DRIVE}/{NOME}.pt')